# Z24 experiment runner

This notebook contains no model logic. Select one config, then run all cells. Attach the dataset containing `inputs.npy` and `labels.npy`; the runner locates it under `/kaggle/input`. Run each experiment in a fresh Kaggle session so TensorFlow and PyTorch do not retain each other's GPU memory.

In [ ]:
# Cell 1 - settings you may change (enable Internet in Kaggle Settings)
from pathlib import Path

REPO_URL = 'https://github.com/tranvanphuongdevdream-web/shm_ml.git'
BRANCH = 'master'
EXPERIMENT_ID = 'tsai_001'  # dcnn_001 | dcnn_002 | tsai_001
OUTPUT_DIR = Path('/kaggle/working/results')
REPO_DIR = Path('/kaggle/working/shm_ml_online')

assert EXPERIMENT_ID in {'dcnn_001', 'dcnn_002', 'tsai_001'}


In [ ]:
# Cell 2 - clone once; reuse the local clone during this Kaggle session
import importlib
import os
import subprocess
import sys

git_environment = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def run_git(arguments):
    return subprocess.run(
        arguments, check=True, timeout=180, env=git_environment,
    )

if (REPO_DIR / '.git').is_dir():
    print('Reusing repository already cloned in this session.', flush=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
else:
    print('Cloning repository...', flush=True)
    run_git(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(REPO_DIR)])

commit = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True,
).strip()
print(f'Repository ready: {REPO_DIR} (commit {commit})', flush=True)
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
sys.path.insert(0, str(REPO_DIR))
from src.data import z24_dataset
INPUTS_PATH, LABELS_PATH = z24_dataset.resolve_data_source()
print('Loaded data module:', z24_dataset.__file__)
print('Dataset inputs:', INPUTS_PATH)
print('Dataset labels:', LABELS_PATH)


In [ ]:
# Cell 3 - install only the dependencies needed by the selected experiment
import hashlib
import importlib
import json
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
from packaging.version import Version
from urllib.request import urlopen

def installed_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

def install_from_pypi(item):
    # Download a compatible wheel at runtime; nothing is bundled with the repository.
    name = item.name.lower()
    exact = [part.version for part in item.specifier if part.operator == '==']
    metadata_url = (
        f'https://pypi.org/pypi/{name}/{exact[0]}/json' if exact
        else f'https://pypi.org/pypi/{name}/json'
    )
    print(f'  Checking PyPI: {metadata_url}', flush=True)
    try:
        with urlopen(metadata_url, timeout=15) as response:
            metadata = json.load(response)
    except Exception as error:
        raise RuntimeError(
            f'Cannot reach PyPI for {item}. Check Kaggle Settings > Internet; '
            f'URL: {metadata_url}; cause: {error}'
        ) from error
    if exact:
        release = exact[0]
        files = metadata['urls']
    else:
        valid = [
            Version(value) for value in metadata['releases']
            if item.specifier.contains(value) and not Version(value).is_prerelease
        ]
        if not valid:
            raise RuntimeError(f'No PyPI release satisfies {item}')
        release = str(max(valid))
        files = metadata['releases'][release]
    tag_priority = {tag: index for index, tag in enumerate(sys_tags())}
    wheels = []
    for file in files:
        if not file['filename'].endswith('.whl'):
            continue
        _, _, _, wheel_tags = parse_wheel_filename(file['filename'])
        compatible = wheel_tags.intersection(tag_priority)
        if compatible:
            wheels.append((min(tag_priority[tag] for tag in compatible), file))
    if not wheels:
        raise RuntimeError(f'No compatible wheel for {name}=={release} on this Kaggle runtime')
    file = min(wheels, key=lambda entry: entry[0])[1]
    print(f'  Downloading {file["filename"]} ({file["size"] / 1024**2:.1f} MiB)', flush=True)
    with tempfile.TemporaryDirectory(prefix='shm-tsai-', dir='/kaggle/working') as directory:
        destination = Path(directory) / file['filename']
        digest = hashlib.sha256()
        received = 0
        started = time.monotonic()
        try:
            with urlopen(file['url'], timeout=20) as source, destination.open('wb') as target:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)
                    digest.update(chunk)
                    received += len(chunk)
                    print(f'  Downloaded {received / 1024**2:.1f} MiB', flush=True)
                    if time.monotonic() - started > 120:
                        raise TimeoutError('Wheel download exceeded 120 seconds')
        except Exception as error:
            raise RuntimeError(
                f'Cannot download {file["filename"]} from {file["url"]}: {error}'
            ) from error
        if digest.hexdigest() != file['digests']['sha256']:
            raise RuntimeError(f'SHA-256 mismatch for {file["filename"]}')
        print(f'  Installing {file["filename"]} without changing PyTorch...', flush=True)
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
            '--disable-pip-version-check', str(destination),
        ], check=True, timeout=120)

if EXPERIMENT_ID == 'tsai_001':
    tsai_package_names = {
        'scikit-learn', 'fastai', 'imbalanced-learn', 'pyts', 'psutil', 'tsai',
    }
    parsed = [
        Requirement(line.strip())
        for line in (REPO_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    ]
    selected = [item for item in parsed if item.name.lower() in tsai_package_names]
    if {item.name.lower() for item in selected} != tsai_package_names:
        raise ValueError('requirements.txt is missing a tsai runtime dependency')
    torch_before = installed_version('torch')
    if torch_before is None:
        raise RuntimeError('Kaggle PyTorch is missing. Select a GPU runtime and restart the session.')
    required_updates = [
        item for item in selected
        if installed_version(item.name) is None
        or not item.specifier.contains(installed_version(item.name), prereleases=True)
    ]
    print('Kaggle runtime packages:', flush=True)
    for item in selected:
        print(f'  {item.name}: {installed_version(item.name) or "missing"}', flush=True)
    for position, item in enumerate(required_updates, start=1):
        print(f'[{position}/{len(required_updates)}] Installing {item} from PyPI...', flush=True)
        install_from_pypi(item)
    importlib.invalidate_caches()
    unresolved = [
        item for item in selected
        if installed_version(item.name) is None
        or not item.specifier.contains(installed_version(item.name), prereleases=True)
    ]
    if unresolved:
        raise RuntimeError(f'Unsatisfied dependencies after installation: {unresolved}')
    if installed_version('torch') != torch_before:
        raise RuntimeError(
            f'PyTorch changed unexpectedly: {torch_before} -> {installed_version("torch")}. '
            'Restart the Kaggle session before continuing.'
        )
    module_names = {
        'scikit-learn': 'sklearn', 'imbalanced-learn': 'imblearn',
    }
    for item in selected:
        importlib.import_module(module_names.get(item.name.lower(), item.name.lower()))
    from fastai.metrics import accuracy  # noqa: F401
    from tsai.data.core import TSClassification  # noqa: F401
    from tsai.data.validation import combine_split_data  # noqa: F401
    from tsai.tslearner import TSClassifier  # noqa: F401
    import tsai.inference  # noqa: F401
    print('Validated tsai imports successfully.', flush=True)
print('Dependencies ready for:', EXPERIMENT_ID, flush=True)


In [ ]:
# Cell 4 - run training directly so Kaggle displays every progress log
import time
from datetime import datetime

existing_runs = {path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*') if path.is_dir()}
started_at = time.perf_counter()
print(f'[{datetime.now():%H:%M:%S}] Starting training', flush=True)
print(f'Experiment: {EXPERIMENT_ID}', flush=True)
print('Live progress will appear below. The first GPU graph compilation may take a few minutes.', flush=True)
from src import run_experiment
importlib.reload(run_experiment)
run_experiment.run('train', EXPERIMENT_ID)
elapsed_minutes = (time.perf_counter() - started_at) / 60
print(f'[{datetime.now():%H:%M:%S}] Training finished in {elapsed_minutes:.2f} minutes', flush=True)
new_runs = [
    path for path in OUTPUT_DIR.glob(f'{EXPERIMENT_ID}_*')
    if path.is_dir() and path not in existing_runs
]
if len(new_runs) != 1:
    raise RuntimeError(f'Expected one new result directory, found: {new_runs}')
RUN_DIR = new_runs[0]
print('Results:', RUN_DIR)
print('Download ZIP:', RUN_DIR.with_suffix('.zip'))


In [ ]:
# Cell 5 - metrics and training benchmark for this run
import json
import pandas as pd
from IPython.display import display

if 'RUN_DIR' not in globals():
    raise RuntimeError('Cell 4 has not completed; no result is available to display.')
metrics = pd.read_csv(RUN_DIR / 'split_metrics.csv', index_col='split')
benchmark = json.loads((RUN_DIR / 'benchmark_summary.json').read_text(encoding='utf-8'))
if benchmark.get('pipeline') != EXPERIMENT_ID:
    raise RuntimeError(
        f'RUN_DIR belongs to {benchmark.get("pipeline")}, not {EXPERIMENT_ID}. Run Cell 4 first.'
    )
print('Accuracy, macro precision, macro recall, and macro F1:')
display(metrics.style.format('{:.2%}'))

benchmark_rows = [
    ('Experiment', benchmark['pipeline']),
    ('Total training time (min)', round(benchmark['training_seconds_total'] / 60, 2)),
    ('Epochs completed', benchmark['epochs_completed']),
    ('Mean epoch time (s)', round(benchmark['mean_epoch_seconds'], 2)),
    ('Mean epoch time, excluding first (s)', round(benchmark['mean_epoch_seconds_excluding_first'], 2)),
    ('Train samples per second', round(benchmark['effective_train_samples_per_second'], 2)),
    ('Model parameters', benchmark['model_parameters']),
    ('Batch size', benchmark['batch_size']),
    ('GPU', ', '.join(benchmark['gpu_names']) or 'CPU'),
    ('Precision policy', benchmark['precision_policy']),
]
print('Training benchmark:')
display(pd.DataFrame(benchmark_rows, columns=['Metric', 'Value']).set_index('Metric'))


In [ ]:
# Cell 6 - learning curves, split comparison, and test confusion matrix
import matplotlib.pyplot as plt
from IPython.display import Image

history = pd.read_csv(RUN_DIR / 'history.csv')
if not history.empty:
    epoch = history['epoch'] if 'epoch' in history else range(1, len(history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for axis, title, columns in (
        (axes[0], 'Loss by epoch', ('loss', 'val_loss', 'train_loss', 'valid_loss')),
        (axes[1], 'Accuracy by epoch', ('accuracy', 'val_accuracy')),
    ):
        available = [column for column in columns if column in history]
        for column in available:
            axis.plot(epoch, history[column], label=column)
        if available:
            axis.set_title(title)
            axis.set_xlabel('Epoch')
            axis.grid(True, alpha=0.3)
            axis.legend()
        else:
            axis.set_visible(False)
    fig.tight_layout()
    plt.show()

axis = metrics[['accuracy', 'f1_macro']].plot.bar(figsize=(8, 4), rot=0)
axis.set_title('Performance by split')
axis.set_ylabel('Score')
axis.set_ylim(0, 1)
axis.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Test confusion matrix:')
display(Image(filename=str(RUN_DIR / 'test_confusion_matrix.png'), width=850))
